# nb03 — backfill(과거 발표 조회) 가능 깊이

재훈련·백필 데이터 확보 가능성을 판단하기 위해, 과거 12z 발표를 하루~3일 간격으로
400일 전까지 찍어 조회 가능 여부를 기록했다 (hf=24 고정, 캐시 재생).

In [1]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
import numpy as np
import pandas as pd
import probe_lib as pl
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

def std_val(grp, nwp, nm, tmfc, hf, x, y, data="U", level=None):
    """캐시된 신규 std 응답에서 첫 값 (캐시에 없으면 실호출 1회)."""
    b = pl.fetch_std(grp, nwp, nm, tmfc, hf, x, y, data=data, level=level)
    if not b or "ERROR" in b:
        return None
    v = [float(t) for l in b.splitlines()
         if l.strip() and not l.startswith("#") for t in l.split()]
    return v[0] if v else None

TMFC = "2026070312"   # 본 연구의 기준 발표 (2026-07-03 12z) -- 캐시 고정
print("호출 예산 상태:", pl.budget_status())

호출 예산 상태: {'real_calls': 1903, 'hard_cap': 10000, 'remaining': 8097}


In [2]:
ar = pd.read_csv("results/archive_scan.csv")
ar["date"] = pd.to_datetime(ar.tmfc.astype(str).str[:8])
for m, g in ar.groupby("model"):
    okd = g[g.ok == 1].date
    ng = g[(g.ok == 0) & (g.date >= okd.min())]
    print(f"{m}: 가장 오래된 가용 발표 = {okd.min().date()}  (가용 {len(okd)}건, "
          f"가용 구간 안 결측 {len(ng)}건)")

L010: 가장 오래된 가용 발표 = 2026-02-09  (가용 69건, 가용 구간 안 결측 0건)
NE57: 가장 오래된 가용 발표 = 2026-01-19  (가용 76건, 가용 구간 안 결측 0건)
R030: 가장 오래된 가용 발표 = 2026-02-09  (가용 69건, 가용 구간 안 결측 0건)


**결과 — 아카이브는 서비스 개시일부터 전량, 결측 없음**

| 모델 | 개시일(문서) | 실측 최고(最古) tmfc | 깊이(2026-07-04 기준) |
|---|---|---|---|
| 전구 NE57 | 2026-01-19 | 2026-01-19 12z ✓ | 약 166일 |
| 지역 R030 | 2026-02-09 | 2026-02-09 12z ✓ | 약 145일 |
| 국지 L010 | 2026-02-09 | 2026-02-09 12z ✓ | 약 145일 |

개시일 이전 날짜는 전부 응답 없음 → 신모델 이름으로는 구모델 과거 자료가 제공되지 않는다.

In [3]:
# 가장 오래된 발표에서도 hf 꼬리(최대 리드)와 4개 발표주기가 온전한가
tc = pd.read_csv("results/archive_tail_cycle.csv")
print(tc.to_string(index=False))
print()
print("전부 ok=1 -> 아카이브는 지평 축약 없이, 4주기 모두 보존")

model       tmfc  hf  kind  ok
 NE57 2026011912 135  tail   1
 NE57 2026011912 285  tail   1
 NE57 2026011912 288  tail   1
 NE57 2026012000  24 cycle   1
 NE57 2026012006  24 cycle   1
 NE57 2026012018  24 cycle   1
 NE57 2026041500  24 cycle   1
 NE57 2026041506  24 cycle   1
 NE57 2026041518  24 cycle   1
 R030 2026020912  72  tail   1
 R030 2026020912 119  tail   1
 R030 2026020912 120  tail   1
 R030 2026021000  24 cycle   1
 R030 2026021006  24 cycle   1
 R030 2026021018  24 cycle   1
 R030 2026041500  24 cycle   1
 R030 2026041506  24 cycle   1
 R030 2026041518  24 cycle   1
 L010 2026020912  24  tail   1
 L010 2026020912  47  tail   1
 L010 2026020912  48  tail   1
 L010 2026021000  24 cycle   1
 L010 2026021006  24 cycle   1
 L010 2026021018  24 cycle   1
 L010 2026041500  24 cycle   1
 L010 2026041506  24 cycle   1
 L010 2026041518  24 cycle   1

전부 ok=1 -> 아카이브는 지평 축약 없이, 4주기 모두 보존


## 추가 추적 — '1h 전구 자료'가 사라진 정확한 시점

현행 운영 DB 는 07-03 이전까지 전구 1h 자료를 받아왔다. 신규 std 아카이브에는 1h 가 아예 없으므로
(개시일부터 3h 간격만), 구 엔드포인트가 읽던 파일을 날짜별로 역추적했다.

In [4]:
# 구 pt 로 비3배수 hf=25 를 과거 날짜에 요청 -- 응답 헤더의 fname 이 결정적 증거
for d in ["20260601", "20260625", "20260630", "20260701", "20260702", "20260703"]:
    b = pl.fetch(pl.URL_OLD_PT, {"group": "KIMG", "nwp": "NE57", "data": "U",
        "name": "t2m,dswrsfc", "tmfc": d + "12", "hf": "25",
        "lat": "33.3284", "lon": "126.8366", "disp": "A", "help": "0"})
    has = b and any(l.strip() and not l.startswith("#") for l in b.splitlines())
    fn = next((l.split("/")[-1].split(",")[0] for l in (b or "").splitlines() if "fname" in l), "")
    print(f"{d} 12z hf=25(1h): {'있음' if has else '없음':3s}  파일={fn}")

20260601 12z hf=25(1h): 있음   파일=g576_v091_glob_sfc.ft025.2026060112.nc
20260625 12z hf=25(1h): 있음   파일=g576_v091_glob_sfc.ft025.2026062512.nc


20260630 12z hf=25(1h): 있음   파일=g576_v091_glob_sfc.ft025.2026063012.nc
20260701 12z hf=25(1h): 없음   파일=g576_v091_glob_etc.2byte.ft025.2026070112.nc
20260702 12z hf=25(1h): 없음   파일=g576_v091_glob_etc.2byte.ft025.2026070212.nc
20260703 12z hf=25(1h): 없음   파일=g576_v091_glob_etc.2byte.ft025.2026070312.nc


**07-01 정책 변경의 실체가 여기서 드러난다**

- 06-30 12z 까지: `g576_v091_glob_sfc.ftNNN` — **1시간 간격** 별도 파일이 존재 (구 엔드포인트가 이걸 읽음)
- 07-01 12z 부터: 그 파일이 사라지고 `g576_v091_glob_etc.2byte.ftNNN` — **3시간 간격**만 생성

즉 "07-03부터 1h 소실"로 관찰됐던 사건의 원인은 **07-01부로 전구 1h 후처리 파일(glob_sfc) 생산이
중단**된 것이다. 과거 1h 자료(glob_sfc, ~06-30)는 지금도 구 엔드포인트로 백필할 수 있다.